# Chapter 6: Growth and Ideas

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/quarcs-lab/macro102-site/blob/main/notebooks/ch06-growth-and-ideas.ipynb)

*Data Science and Macroeconomic Models: A Python Companion to Intermediate Macroeconomics* · Carlos Mendez

> **How to use this notebook.** In Colab, click *Runtime → Run all*. After that, every model
> figure has live sliders: move one and the model is solved again on the spot. On the book
> website the same figures come with one precomputed slider. Each section ends with a short
> **Your turn** task, and you can try it in the empty code cell below the task.

## Learning goals

The Solow model of Chapter 5 could not explain sustained growth. This chapter adds the missing
engine: new **ideas**. By the end of the chapter you will be able to:

- explain why ideas are **nonrival** and why nonrivalry leads to **increasing returns**;
- show with a fixed cost why prices must lie above marginal cost if anyone is to pay for research;
- solve the **Romer model** and compute its long-run growth rate $g_y^* = \gamma\bar{n}$;
- tell **growth effects** from **level effects** in experiments with the Romer model and the
  combined Solow–Romer model;
- read long-run data on commodity prices, U.S. growth and growth accounting;
- simulate a model year by year with a `for` loop, draw ratio-scale charts in Plotly, and compute
  growth rates over periods with pandas.

## Figures in this chapter

The figures follow the order of the textbook chapter. The second column shows which textbook
figure each notebook figure replicates.

| Figure | Textbook figure | Type | What you can change |
|---|---|---|---|
| Figure 1 | 6.1 A fixed cost leads to increasing returns | model | fixed cost, scale of production |
| Figure 2 | 6.2 The balanced growth path in the Romer model | model | importance of ideas, population growth, linear or ratio scale |
| Figure 3 | 6.3 A rise in research productivity | model | new research productivity, linear or ratio scale |
| Figure 4 | 6.4 A rise in the research share | model | new research share, linear or ratio scale |
| Figure 5 | 6.5 The real price of industrial commodities | data | the six commodities in the index (legend) |
| Figure 6 | 6.6 A slowdown in U.S. economic growth? | data | number of years in the average |
| Figure 7 | 6.7 The word processor production function | model | fixed cost, price per copy |
| Figure 8 | 6.8 Output after a rise in the investment rate | model | old and new investment rate, linear or ratio scale |

## Setup

We first load the Python libraries we need. You only have to run this cell once.

> **Python note.** `import` loads a *library*, a toolbox written by other people. We give each
> library a short nickname, for example `np` for NumPy, so that the code stays short. Text after
> a `#` is a *comment*: Python ignores it, and it is there for human readers. The `try ... except`
> block runs one Colab-only line and simply skips it anywhere else.

In [1]:
# Libraries used in this chapter
import numpy as np                              # numbers and arrays
import pandas as pd                             # data tables
import plotly.express as px                     # quick charts from data tables
import plotly.graph_objects as go               # charts built piece by piece (model figures)
import plotly.io as pio
from plotly.subplots import make_subplots       # several panels in one figure
from ipywidgets import interact, FloatSlider, ToggleButtons, Dropdown

# One light blue/green look for every chart
BLUE, TEAL, GREEN, GRAY = "#16528c", "#008070", "#7fb23a", "#8a96a3"
pio.templates.default = "plotly_white"
pio.templates["plotly_white"].layout.update(width=860, height=480)   # same size in Colab and on the website
px.defaults.color_discrete_sequence = [BLUE, TEAL, GREEN, "#5aa0d8", "#0b4f47"]
WIDE = {"description_width": "initial"}         # show the full slider labels

# Colab only: let Plotly charts appear inside the interactive widgets
try:
    import google.colab
    google.colab.output.enable_custom_widget_manager()
except ImportError:
    pass                                        # not running in Colab: nothing to do

# The data files of this chapter live in the book's GitHub repository
DATA_URL = "https://raw.githubusercontent.com/quarcs-lab/macro102-site/main/data/ch06/"

The next cell defines a helper that only the book website needs. Run it and move on.

In [2]:
# @title Helper for the website versions (run this cell; you do not need to read it)
def slider_figure(make_figure, name, values, start, label=None):
    """Precompute make_figure(name=value) for every value and join the results with one slider.

    The website cannot run Python, so each interactive figure also gets this precomputed version.
    """
    figures = [make_figure(**{name: v}) for v in values]
    first = values.index(start)
    fig = go.Figure(layout=figures[first].layout)
    owner = []                                  # which precomputed figure each trace comes from
    for i, f in enumerate(figures):
        for trace in f.data:
            trace.visible = (i == first)
            fig.add_trace(trace)
            owner.append(i)
    steps = []
    for i, f in enumerate(figures):
        layout = {"title": f.layout.title.to_plotly_json(),
                  "annotations": [a.to_plotly_json() for a in f.layout.annotations],
                  "shapes": [s.to_plotly_json() for s in f.layout.shapes]}
        visible = [o == i for o in owner]
        steps.append(dict(method="update", label=str(values[i]), args=[{"visible": visible}, layout]))
    fig.update_layout(sliders=[dict(active=first, steps=steps, pad=dict(t=60),
                                    currentvalue=dict(prefix=(label or name) + " = "))])
    return fig

## The model in code

Paul Romer divides economic goods into **objects** and **ideas**. Objects are things: land, oil,
machines, workers. Ideas are instructions for using objects, such as the design of a phone or the
formula of a drug. The key difference is that ideas are **nonrival**: my use of the quadratic
formula does not leave less of it for you. So the same idea can serve any number of workers and
factories at once.

**Nonrivalry gives increasing returns.** Write output as $Y_t = A_tK_t^{1/3}L_t^{2/3}$ (6.1),
where $A_t$ is the stock of ideas. Doubling the objects $K$ and $L$ doubles output, because the
new factory can use the same ideas. Doubling the ideas as well therefore *more* than doubles output.

> **Python note.** `def` creates a *function*: a named recipe that takes inputs (the
> *arguments*) and gives back a result with `return`. The text in triple quotes is a short
> description of what the function does. `**` means "to the power of". An *f-string* such as
> `f"{ratio:.2f}"` inserts the value of a variable into a piece of text; `:.2f` means "two decimals".

In [3]:
def output_with_ideas(K, L, A):
    """Production with objects and ideas: Y = A K^(1/3) L^(2/3)   (equation 6.1)."""
    return A * K ** (1 / 3) * L ** (2 / 3)


base = output_with_ideas(K=1, L=1, A=1)
print(f"Double K and L only:       output x {output_with_ideas(K=2, L=2, A=1) / base:.2f}")
print(f"Double K, L and A:         output x {output_with_ideas(K=2, L=2, A=2) / base:.2f}")
# How many 100-word paragraphs can be written with 20,000 words? 20,000^100 = 10^(100 log10 20,000)
print(f"Possible idea paragraphs:  10 to the power {100 * np.log10(20000):.1f}")

Double K and L only:       output x 2.00
Double K, L and A:         output x 4.00
Possible idea paragraphs:  10 to the power 430.1


**The Romer model** has no capital, so that we can see the role of ideas clearly. It has five
equations and five unknowns (textbook Table 6.1):

| Equation | Name |
|---|---|
| $Y_t = A_t^{\gamma}L_{yt}$ (6.2) | output: workers $L_{yt}$ use the whole stock of ideas $A_t$ |
| $\Delta A_{t+1} = \bar{z}L_{at}$ (6.3) | new ideas: each researcher finds $\bar{z}$ ideas a year |
| $L_{yt} + L_{at} = N_t$ (6.4) | people either make goods or do research |
| $\Delta N_{t+1}/N_t = \bar{n}$ (6.5) | the population grows at rate $\bar{n}$ |
| $L_{at} = \bar{\ell}N_t$ (6.6) | a constant share $\bar{\ell}$ of the population does research |

The parameter $\gamma$ measures how important ideas are, and it equals the degree of increasing
returns. Output per person is $y_t = A_t^{\gamma}(1-\bar{\ell})$ (6.7). It depends on the **total**
stock of ideas, not on ideas per person: that is nonrivalry at work.

On the **balanced growth path (BGP)**, where all growth rates are constant, the stock of ideas grows
at the rate of population, $g_A^* = \bar{n}$, and is $A_t^* = \bar{z}\bar{\ell}N_t/\bar{n}$ (6.8).
Putting this into (6.7) gives the two key results:

$$y_t^* = (1-\bar{\ell})\left(\frac{\bar{z}\bar{\ell}N_t}{\bar{n}}\right)^{\gamma} \quad (6.10) \qquad\qquad g_y^* = \gamma\bar{n} \quad (6.11)$$

### Simulating the Romer model year by year

To draw the experiments of the chapter, we let a computer apply the equations one year at a time.
The function below starts on the BGP in 2000 and changes $\bar{z}$ and $\bar{\ell}$ in `change_year`.
It returns output per person and, for comparison, the BGP level for the parameters in force in each
year. Both are scaled so that $y_{2000} = 100$, as in the textbook.

> **Python note.** `np.arange(2000, 2091)` gives the years 2000, 2001, ..., 2090. A `for` loop
> repeats the indented lines once for each year, and `if ... else` chooses between two cases.
> `y_path.append(...)` adds a number to the end of a list. Writing `gamma=0.6` gives the argument a
> *default value*, used whenever we do not pass one.

In [4]:
def simulate_romer(z_old=1.0, z_new=1.0, l_old=0.1, l_new=0.1, gamma=0.6, n=0.05,
                   change_year=2030, first_year=2000, last_year=2090):
    """Romer model (equations 6.2-6.6) year by year; z and l change to z_new, l_new in change_year."""
    years = np.arange(first_year, last_year + 1)
    N = 1.0                                         # population in the first year
    A = z_old * l_old * N / n                       # start with the BGP stock of ideas, equation (6.8)
    y_path, bgp_path = [], []
    for year in years:
        if year < change_year:
            z, l = z_old, l_old
        else:
            z, l = z_new, l_new
        y_path.append(A ** gamma * (1 - l))                  # output per person, equation (6.7)
        bgp_path.append((1 - l) * (z * l * N / n) ** gamma)  # BGP level, equation (6.10)
        A = A + z * l * N                           # new ideas from the researchers, equation (6.3)
        N = N * (1 + n)                             # population growth, equation (6.5)
    scale = 100 / y_path[0]                         # normalization: y = 100 in the first year
    return years, scale * np.array(y_path), scale * np.array(bgp_path)

A quick check. With yearly steps, output per person on the BGP grows by the factor $(1+\bar{n})^{\gamma}$
each year, which is very close to $1 + \gamma\bar{n}$.

In [5]:
years, y, y_bgp = simulate_romer(gamma=0.7, n=0.05)
print(f"Simulated growth of y in 2001:  {100 * (y[1] / y[0] - 1):.2f}% a year")
print(f"Equation (6.11), gamma * n:     {100 * 0.7 * 0.05:.2f}% a year")
print(f"Two textbook cases: gamma = 2, n = 1%: {2 * 1}%;  gamma = 1/2, n = 4%: {0.5 * 4:.0f}%")

Simulated growth of y in 2001:  3.47% a year
Equation (6.11), gamma * n:     3.50% a year
Two textbook cases: gamma = 2, n = 1%: 2%;  gamma = 1/2, n = 4%: 2%


## Figure 1: A fixed cost leads to increasing returns

**Goal.** See why a fixed cost of research turns a constant-returns technology into one with
increasing returns.

Once a new antibiotic has been invented, making it is ordinary production: each \$10 buys one
dose, so $Y = X/10$ when $X$ dollars are spent. Doubling spending doubles the doses, and the
**average product** $Y/X$ is constant. But first someone must pay the research cost
$\bar{F} = \$2.5$ billion. Including it, $Y = (X-\bar{F})/10$ for $X > \bar{F}$, and

$$\frac{Y}{X} = \frac{1}{10}\left(1 - \frac{\bar{F}}{X}\right),$$

which **rises** with the scale of production $X$.

In the chart, $X$ is in billions of dollars and $Y$ in millions of doses, so $Y = 100\,(X - \bar{F})$.

In [6]:
def doses(X, F=0.0):
    """Antibiotic doses (millions) from spending X (billions of $) with a fixed cost F: Y = (X - F)/10."""
    return 100 * np.maximum(X - F, 0)               # zero until the fixed cost is paid

> **Python note.** `np.maximum(X - F, 0)` takes, for each number, the larger of $X - \bar{F}$
> and 0, so output is zero until the fixed cost is covered. `make_subplots(rows=1, cols=2)`
> creates a figure with two panels side by side, and `row=..., col=...` tells `add_trace` where to draw.

In [7]:
def fixed_cost(F=2.5, X=7.2):
    """Figure 6.1: constant returns (a) and the same technology with a fixed cost F (b)."""
    x = np.linspace(0, 10, 201)
    half = X / 2                                     # the smaller of the two scales we compare
    fig = make_subplots(rows=1, cols=2, subplot_titles=("(a) Constant returns: Y = X/10",
                                                        f"(b) Fixed cost F̄ = {F} billion"))
    fig.add_trace(go.Scatter(x=x, y=doses(x), line=dict(color=TEAL, width=3)), row=1, col=1)
    fig.add_trace(go.Scatter(x=x, y=doses(x, F), line=dict(color=GREEN, width=3)), row=1, col=2)
    for scale in [half, X]:                          # a dot in (a) and an average-product ray (slope Y/X) in (b)
        fig.add_trace(go.Scatter(x=[scale], y=[doses(scale)], marker=dict(color="black", size=10)), row=1, col=1)
        fig.add_trace(go.Scatter(x=[0, scale], y=[0, doses(scale, F)], line=dict(color="black", dash="dash")), row=1, col=2)
    if F > 0:
        fig.add_annotation(x=2.6, y=620, xref="x2", yref="y2", showarrow=False, font=dict(size=13),
                           text="The average product Y/X<br>(slope of a dashed ray)<br>rises with scale")
    if doses(half, F) > 0:
        gain = f"×{doses(X, F) / doses(half, F):.2f}"
    else:
        gain = "from zero"
    fig.update_xaxes(title_text="Input, X (billions of dollars)", range=[0, 10])
    fig.update_yaxes(title_text="Output, Y (millions of doses)", range=[0, 1000], row=1, col=1)
    fig.update_yaxes(range=[0, 800], row=1, col=2)
    fig.update_layout(title=f"Doubling X from {half:.1f} to {X:.1f} billion: output ×2 in (a), {gain} in (b)",
                      showlegend=False)
    return fig

Now the interactive version. Move the sliders and compare the two panels.

> **Python note.** `interact` turns each argument of a function into a slider and calls the
> function again every time you move one. The `;` at the end only hides an extra line of text.

In [8]:
interact(fixed_cost,
         F=FloatSlider(value=2.5, min=0.0, max=5.0, step=0.5, description="fixed cost F̄ (billions)", style=WIDE),
         X=FloatSlider(value=7.2, min=5.0, max=10.0, step=0.2, description="scale of production X", style=WIDE));

In [9]:
# @title Website version: Figure 1 with a precomputed slider for the fixed cost
slider_figure(fixed_cost, "F", [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5], start=2.5, label="fixed cost F̄ (billions)")

**What to notice.**

- In panel (a) both black dots lie on a straight line through the origin: doubling the input exactly doubles
  the doses. That is constant returns to scale.
- In panel (b) the first \$2.5 billion produce nothing. Doubling the input from 3.6 to 7.2 billion
  multiplies output by more than four, and the dashed ray to the larger scale is steeper: the
  average product rises with scale.
- Set $\bar{F} = 0$ and panel (b) becomes panel (a). The increasing returns come only from the fixed cost.

### Key numbers: pricing at marginal cost

If the firm had to sell each dose at its marginal cost of \$10, the revenue would just cover the
production cost of the doses, and nothing would be left to pay for the research.

In [10]:
X = 5.0                                              # spend 2.5 billion on research and 2.5 on production
Y = doses(X, F=2.5)                                  # millions of doses
revenue = Y * 10 / 1000                              # billions of dollars at a price of $10 a dose
print(f"Spending {X} billion gives {Y:.0f} million doses")
print(f"Revenue at marginal cost: {revenue:.1f} billion; profit: {revenue - X:.1f} billion")

Spending 5.0 billion gives 250 million doses
Revenue at marginal cost: 2.5 billion; profit: -2.5 billion


The firm loses exactly the research cost. This is why new ideas need a gap between price and marginal
cost (from patents, for example), or another way of paying for research, such as public funding or prizes.

**Your turn.** At what price per dose would the firm in the cell above just break even? Compute it in the
empty cell below. Then keep $X = 7.2$ and set $\bar{F}$ to 1.0 and then to 3.5. How does the output gain
in the title of panel (b) change, and why?

<details><summary>Hint</summary>

The firm breaks even when the price times the number of doses equals the total spending of 5 billion.
For the second part, the gain is $(7.2-\bar{F})/(3.6-\bar{F})$: the closer the smaller scale is to the fixed
cost, the less output it produces.

</details>

In [11]:
# Your turn: try it here

## Figure 2: The balanced growth path in the Romer model

**Goal.** See that the Romer model produces **sustained** growth in output per person, at the rate
$g_y^* = \gamma\bar{n}$.

More people means more researchers, and more researchers find more ideas. Because ideas are nonrival,
every new idea raises the income of *everyone*. So as long as the population grows, output per person
grows too, at a constant rate.

On a **ratio scale** (a logarithmic axis), equal distances mean equal *percentage* changes, so a
variable that grows at a constant rate is a straight line, and the slope of the line is the growth rate.
We will use the same vertical axis in Figures 2, 3, 4 and 8, so we write it once as a small function.

> **Python note.** On a log axis Plotly wants the axis limits as powers of ten, which is why we write
> `np.log10(...)`. `tickvals` sets the numbers printed on the axis: each one is twice the one below.

In [12]:
TICKS = [100, 200, 400, 800, 1600, 3200, 6400]           # no label at the bottom corner


def vertical_axis(fig, scale):
    """Output per person on a ratio (log) or linear vertical axis, with fixed ranges."""
    if scale == "ratio":
        fig.update_yaxes(type="log", range=[np.log10(50), np.log10(6400)], tickvals=TICKS)
    else:
        fig.update_yaxes(type="linear", range=[0, 6400])
    fig.update_yaxes(title_text=f"Output per person, y ({scale} scale)")
    fig.update_xaxes(title_text="Year", range=[2000, 2095])

In [13]:
def romer_bgp(gamma=0.7, n=0.05, scale="ratio"):
    """Figure 6.2: output per person on the balanced growth path of the Romer model (y_2000 = 100)."""
    years, y, y_bgp = simulate_romer(gamma=gamma, n=n, last_year=2080)
    growth = 100 * (y[1] / y[0] - 1)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=years, y=y, name="Output per person, y", line=dict(color=BLUE, width=3)))
    vertical_axis(fig, scale)
    fig.update_layout(title=f"Constant growth of {growth:.2f}% a year (γ·n = {100 * gamma * n:.2f}%); "
                            f"y in 2080 = {y[-1]:,.0f}", showlegend=False)
    return fig

In [14]:
interact(romer_bgp,
         gamma=FloatSlider(value=0.7, min=0.1, max=1.0, step=0.1, description="importance of ideas γ", style=WIDE),
         n=FloatSlider(value=0.05, min=0.01, max=0.05, step=0.005, description="population growth n",
                       style=WIDE, readout_format=".3f"),
         scale=ToggleButtons(options=["ratio", "linear"], description="vertical scale"));

In [15]:
# @title Website version: Figure 2 with a precomputed slider for γ and buttons for the vertical scale
fig = slider_figure(romer_bgp, "gamma", [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0], start=0.7,
                    label="importance of ideas γ")
fig.update_layout(updatemenus=[dict(type="buttons", direction="right", x=0.01, y=0.98, xanchor="left",
                                    yanchor="top", buttons=[
    dict(label="ratio scale", method="relayout",
         args=[{"yaxis.type": "log", "yaxis.range": [np.log10(50), np.log10(6400)], "yaxis.tickvals": TICKS,
                "yaxis.title.text": "Output per person, y (ratio scale)"}]),
    dict(label="linear scale", method="relayout",
         args=[{"yaxis.type": "linear", "yaxis.range": [0, 6400], "yaxis.tickvals": None,
                "yaxis.title.text": "Output per person, y (linear scale)"}])])])
fig

**What to notice.**

- On the ratio scale the path is a straight line: output per person grows at the same rate every year.
  With $\gamma = 0.7$ and $\bar{n} = 0.05$ it grows about 3.5 percent a year, and it is more than 15 times
  its 2000 level by 2080.
- Switch to the linear scale: the same path now curves upward, because a constant *percentage*
  growth adds more and more *dollars* each year.
- Growth rises with both $\gamma$ (how important ideas are) and $\bar{n}$ (how fast the number of
  researchers grows). With $\bar{n} = 0$ there would be no long-run growth at all.

**Your turn.** The United States has grown at about 2 percent a year for more than 150 years. Find two
combinations of $\gamma$ and $\bar{n}$ on the sliders that give a growth rate of about 2 percent.

<details><summary>Hint</summary>

You need $\gamma \times \bar{n} \approx 0.02$. Try $\bar{n} = 0.04$ first, then $\bar{n} = 0.025$.

</details>

In [16]:
# Your turn: try it here

## Figure 3: A rise in research productivity

**Goal.** Run the first experiment: in 2030, research productivity $\bar{z}$ rises permanently, for
example because artificial intelligence makes it easier to find new ideas.

**Step 1: the long run.** The growth rate $g_y^* = \gamma\bar{n}$ does not depend on $\bar{z}$, so long-run
growth is unchanged. But the BGP level in (6.10) is proportional to $\bar{z}^{\gamma}$, so the new BGP
is higher by the factor $(\bar{z}'/\bar{z})^{\gamma}$: a **level effect**.

**Step 2: the transition.** In 2030 the stock of ideas has not changed yet, so the economy is
suddenly *below* its new BGP. By the **principle of transition dynamics**, it grows fast at first,
and then more and more slowly as it gets close to the new path.

We reuse `simulate_romer`. The old BGP is simply a simulation in which nothing changes.

In [17]:
def rise_in_z(z_new=8.0, scale="ratio"):
    """Figure 6.3: output per person when research productivity rises from 1 to z_new in 2030."""
    years, y, y_bgp = simulate_romer(z_new=z_new)
    years, y_old, y_old_bgp = simulate_romer()             # no change: the old balanced growth path
    after = years >= 2030                                  # True for 2030 and later
    level = y_bgp[-1] / y_old[-1]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=years[after], y=y_old[after], name="Old BGP", line=dict(color=GRAY, width=3, dash="dash")))
    fig.add_trace(go.Scatter(x=years[after], y=y_bgp[after], name="New BGP", line=dict(color=GREEN, width=3, dash="dash")))
    fig.add_trace(go.Scatter(x=years, y=y, name="Output per person, y", line=dict(color=BLUE, width=3)))
    fig.add_trace(go.Scatter(x=[2093, 2093], y=[y_old[-1], y_bgp[-1]], name=f"Level effect: ×{level:.2f}",
                             mode="lines+markers", line=dict(color="black", width=2)))
    vertical_axis(fig, scale)
    # y[30] is 2030, y[31] is 2031 and y[-1] is the last year, 2090
    fig.update_layout(title=f"New BGP ×{level:.2f} the old | growth {100 * (y[31] / y[30] - 1):.1f}% in 2031, "
                            f"{100 * (y[-1] / y[-2] - 1):.1f}% in 2090 (BGP: {100 * (y[1] / y[0] - 1):.1f}%)")
    return fig

In [18]:
interact(rise_in_z,
         z_new=FloatSlider(value=8.0, min=0.5, max=10.0, step=0.5, description="new research productivity z̄'",
                           style=WIDE),
         scale=ToggleButtons(options=["ratio", "linear"], description="vertical scale"));

In [19]:
# @title Website version: Figure 3 with a precomputed slider for the new research productivity
slider_figure(rise_in_z, "z_new", [0.5, 1.0, 2.0, 4.0, 6.0, 8.0, 10.0], start=8.0, label="new research productivity z̄'")

**What to notice.**

- With $\gamma = 0.6$, an eightfold rise in $\bar{z}$ lifts the BGP by $8^{0.6} \approx 3.48$. Output per person
  jumps onto a steep path: growth is about 22 percent in 2031, then it falls back toward the old
  3 percent as the economy approaches the new BGP.
- The change has a permanent effect on the **level** of income but not on its long-run **growth rate**.
- Set $\bar{z}' = 0.5$: research becomes less productive, growth slows for a while, and the economy
  settles on a lower BGP.

**Your turn.** Suppose research productivity falls by 10 percent (set $\bar{z}' = 0.9$ in the empty
cell below, since the slider moves in steps of 0.5). By what percentage is the new BGP below the old one?

<details><summary>Hint</summary>

Call `rise_in_z(z_new=0.9)` to see the chart, and compare its level effect in the legend with $0.9^{0.6}$.

</details>

In [20]:
# Your turn: try it here

## Figure 4: A rise in the research share

**Goal.** Run the second experiment: in 2030 the economy permanently puts a larger share
$\bar{\ell}$ of its people into research.

**Step 1: the long run.** Again $g_y^* = \gamma\bar{n}$ does not change. The BGP level (6.10) can be
written $y_t^* = (1-\bar{\ell})\,\bar{\ell}^{\gamma}\,(\bar{z}N_t/\bar{n})^{\gamma}$, so $\bar{\ell}$ enters twice:
more researchers mean more ideas ($\bar{\ell}^{\gamma}$), but fewer workers making goods ($1-\bar{\ell}$).
For a research share that is not too large, the first effect wins and the new BGP is higher.

**Step 2: the transition.** In 2030 the stock of ideas is still the same, but fewer people produce
goods. So output per person **falls** on impact. After that, the economy is far below its new BGP
and grows fast to catch up.

In [21]:
def rise_in_l(l_new=0.35, scale="ratio"):
    """Figure 6.4: output per person when the research share rises from 0.03 to l_new in 2030."""
    years, y, y_bgp = simulate_romer(l_old=0.03, l_new=l_new)
    years, y_old, y_old_bgp = simulate_romer(l_old=0.03, l_new=0.03)   # the old balanced growth path
    after = years >= 2030
    level = y_bgp[-1] / y_old[-1]
    impact = 100 * (y[30] / y_old[30] - 1)                 # change in 2030 compared with the old path
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=years[after], y=y_old[after], name="Old BGP", line=dict(color=GRAY, width=3, dash="dash")))
    fig.add_trace(go.Scatter(x=years[after], y=y_bgp[after], name="New BGP", line=dict(color=GREEN, width=3, dash="dash")))
    fig.add_trace(go.Scatter(x=years, y=y, name="Output per person, y", line=dict(color=BLUE, width=3)))
    fig.add_trace(go.Scatter(x=[2093, 2093], y=[y_old[-1], y_bgp[-1]], name=f"Level effect: ×{level:.2f}",
                             mode="lines+markers", line=dict(color="black", width=2)))
    vertical_axis(fig, scale)
    fig.update_layout(title=f"Impact in 2030: {impact:+.0f}% | new BGP ×{level:.2f} the old | "
                            f"growth in 2031: {100 * (y[31] / y[30] - 1):.1f}%")
    return fig

In [22]:
interact(rise_in_l,
         l_new=FloatSlider(value=0.35, min=0.05, max=0.8, step=0.05, description="new research share ℓ̄'", style=WIDE),
         scale=ToggleButtons(options=["ratio", "linear"], description="vertical scale"));

In [23]:
# @title Website version: Figure 4 with a precomputed slider for the new research share
slider_figure(rise_in_l, "l_new", [0.05, 0.1, 0.2, 0.35, 0.5, 0.65, 0.8], start=0.35, label="new research share ℓ̄'")

**What to notice.**

- When $\bar{\ell}$ rises from 0.03 to 0.35, output per person first falls by 33 percent: many workers have
  left goods production, and their new ideas have not arrived yet.
- Then output grows very fast, and in the long run the new BGP is about 2.9 times the old one.
- Move $\bar{\ell}'$ to 0.65 or 0.8. The impact drop is deeper and the level effect *smaller*: too many
  researchers leave too few workers to use the ideas.

### Key numbers: the best research share

The level effect $\dfrac{(1-\bar{\ell}')\,\bar{\ell}'^{\,\gamma}}{(1-\bar{\ell})\,\bar{\ell}^{\gamma}}$ is
**hump shaped** in the new share $\bar{\ell}'$. The table computes it for several shares, starting
from $\bar{\ell} = 0.03$ with $\gamma = 0.6$.

In [24]:
def bgp_level(l, gamma=0.6):
    """The part of the BGP level (6.10) that depends on the research share: (1 - l) l^gamma."""
    return (1 - l) * l ** gamma


rows = []
for l_new in [0.03, 0.1, 0.2, 0.3, 0.35, 0.4, 0.5, 0.65, 0.8]:
    rows.append([l_new, bgp_level(l_new) / bgp_level(0.03)])
print(f"The level effect is largest at l = gamma / (1 + gamma) = {0.6 / 1.6:.3f}")
pd.DataFrame(rows, columns=["new research share", "new BGP / old BGP"]).round(2)

The level effect is largest at l = gamma / (1 + gamma) = 0.375


,new research share,new BGP / old BGP
0,0.03,1.00
1,0.10,1.91
2,0.20,2.57
3,0.30,2.87
4,0.35,2.93
5,0.40,2.93
6,0.50,2.79
7,0.65,2.28
8,0.80,1.48


**Your turn.** Suppose that, with $\gamma = 2/3$, the research share falls from
0.10 to 0.05. Use `bgp_level` with `gamma=2/3` to find how much poorer the economy is on its new BGP.

<details><summary>Hint</summary>

Compute `bgp_level(0.05, gamma=2/3) / bgp_level(0.10, gamma=2/3)`. A result below 1 means a lower BGP.

</details>

In [25]:
# Your turn: try it here

## Figure 5: The real price of industrial commodities

**Goal.** Ask whether a growing world economy must run out of natural resources, using a century of
commodity prices.

The world's supply of aluminum, coal or copper is finite, and in the twentieth century the demand for
them rose enormously. Simple supply and demand would predict rising prices. The Romer model suggests
another possibility: new ideas, such as new ways to find, extract and use resources, can expand
supply faster than demand.

> **Python note.** `pd.read_csv` reads a data file into a *DataFrame*, a table with named
> columns. `.head()` shows its first five rows, which is always a good first look at new data.

In [26]:
prices = pd.read_csv(DATA_URL + "fig6-5.csv")
prices.head()

,year,commodity_index,aluminum_real_1900_100,coal_real_1900_100,copper_real_1900_100,lead_real_1900_100,iron_ore_real_1900_100,zinc_real_1900_100,extended_with_world_bank
0,1900,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,0
1,1901,92.3121,99.6936,99.7363,98.2953,97.8871,70.6218,91.5872,0
2,1902,89.3555,98.4982,105.1100,66.4937,90.9035,75.5896,107.6060,0
3,1903,93.8080,96.3043,113.7790,78.0370,92.5919,76.3423,112.7430,0
4,1904,87.8967,100.9570,99.7638,74.6883,93.0258,62.2125,105.9230,0


Each row is a year. The six `..._real_1900_100` columns are real prices (divided by the U.S. consumer price
index) of aluminum, coal, copper, lead, iron ore and zinc, each set to 100 in 1900. `commodity_index`
combines them with equal weights, and `extended_with_world_bank` marks the years 2021–2025, which were
added from a second source (World Bank prices). The textbook uses proprietary data, so our index differs
somewhat in levels but has the same ups and downs.

> **Python note.** `prices.set_index("year").commodity_index` makes the years the row labels of the index
> column, so that `index[2000]` picks the value for 2000. `add_hline` draws a horizontal line.

In [27]:
index = prices.set_index("year").commodity_index
fig = px.line(prices, x="year", y="commodity_index",
              labels={"year": "Year", "commodity_index": "Equally weighted price index (1900 = 100)"},
              title=f"Real commodity prices fell from 100 in 1900 to {index[2000]:.0f} in 2000, "
                    f"then rose to {index[2011]:.0f} in 2011")
fig.update_traces(line=dict(color=TEAL, width=3))
fig.add_hline(y=100, line_dash="dash", line_color=GRAY)
fig.add_hline(y=20, line_dash="dash", line_color=GRAY)
fig.update_layout(yaxis_range=[0, 140], xaxis_range=[1900, 2026])
fig.show()

**What to notice.**

- Hover over the line. Apart from the World War I spike, prices fell for most of the century, with a
  trough in 2002.
- After 2002 prices rose steeply, which the textbook links to the fast growth of China and India.
  The latest years have fallen back again.

### Key numbers: the fall and the rise

In [28]:
print(f"Peak during World War I, 1916:  {index[1916]:.1f}")
print(f"Lowest value, {index.idxmin()}:            {index.min():.1f}")
print(f"Fall from 1900 to 2000:         factor {index[1900] / index[2000]:.1f} (textbook: almost 5)")
print(f"Rise from 2000 to 2011:         factor {index[2011] / index[2000]:.2f}")
print(f"Latest year, 2025:              {index[2025]:.1f}")

Peak during World War I, 1916:  124.1
Lowest value, 2002:            30.1
Fall from 1900 to 2000:         factor 2.9 (textbook: almost 5)
Rise from 2000 to 2011:         factor 2.43
Latest year, 2025:              58.9


With our data, prices fell by a factor of about 3 between 1900 and 2000, rather than the textbook's
"almost five", but the main point stands: during a century of massive industrialization, the real
price of these raw materials fell rather than rose.

### Inside the index: six commodities

How was the index built? A simple average of the six prices is not a good choice: one commodity
whose price shoots up would dominate it. The index here is a **geometric mean**: the average of the
*logarithms* of the prices, turned back into a price with `np.exp`. Let us check that this rebuilds
the index, and compare it with a simple (arithmetic) mean.

> **Python note.** `prices[COMMODITIES]` keeps only the listed columns. `.mean(axis=1)` averages
> *across* the columns of each row (one average per year), and `.isin([...])` keeps the listed years.

In [29]:
COMMODITIES = ["aluminum_real_1900_100", "coal_real_1900_100", "copper_real_1900_100",
               "lead_real_1900_100", "iron_ore_real_1900_100", "zinc_real_1900_100"]
check = pd.DataFrame({"year": prices.year, "index in the file": prices.commodity_index,
                      "geometric mean": np.exp(np.log(prices[COMMODITIES]).mean(axis=1)),
                      "arithmetic mean": prices[COMMODITIES].mean(axis=1)})
check[check.year.isin([1900, 1950, 2000, 2022, 2025])].round(1)

,year,index in the file,geometric mean,arithmetic mean
0,1900,100.0,100.0,100.0
50,1950,68.0,68.0,83.4
100,2000,34.5,34.5,46.3
122,2022,83.8,83.8,190.8
125,2025,58.9,58.9,85.9


The geometric mean reproduces the index exactly. The arithmetic mean is much higher in 2022 because of
a single commodity. The chart below shows which one. It uses a ratio scale, because the six prices
differ a lot. Click a name in the legend to hide or show that line.

> **Python note.** `.melt(id_vars="year")` turns the commodity columns into rows: the old column names
> go into a new column (`var_name`) and the numbers into another (`value_name`). `.map(NAMES)` replaces
> each column name with a readable name, using the dictionary `NAMES`.

In [30]:
NAMES = {"commodity_index": "Index", "aluminum_real_1900_100": "Aluminum", "coal_real_1900_100": "Coal",
         "copper_real_1900_100": "Copper", "lead_real_1900_100": "Lead",
         "iron_ore_real_1900_100": "Iron ore", "zinc_real_1900_100": "Zinc"}
long = prices.melt(id_vars="year", value_vars=list(NAMES), var_name="series", value_name="price")
long["commodity"] = long.series.map(NAMES)
fig = px.line(long, x="year", y="price", color="commodity", log_y=True,
              color_discrete_sequence=["black", "#5aa0d8", "#7fb23a", "#e0a33b", "#b04a5a", "#16528c", "#008070"],
              labels={"year": "Year", "price": "Real price (1900 = 100, ratio scale)", "commodity": "Commodity"},
              title="Aluminum became far cheaper; coal spiked in 2022")
fig.update_traces(line=dict(width=1.5))
fig.update_traces(line=dict(color="black", width=4), selector=dict(name="Index"))
fig.update_yaxes(tickvals=[5, 10, 20, 50, 100, 200, 500, 1000], range=[np.log10(4), np.log10(1000)])
fig.show()

In [31]:
latest = prices.set_index("year").loc[2025, COMMODITIES]
print("Real prices in 2025 (1900 = 100):")
print(latest.round(1))
print("Coal in 2022:", round(prices.set_index("year").loc[2022, "coal_real_1900_100"], 1))

Real prices in 2025 (1900 = 100):
aluminum_real_1900_100      8.0
coal_real_1900_100        226.8
copper_real_1900_100       72.8
lead_real_1900_100         53.2
iron_ore_real_1900_100     73.6
zinc_real_1900_100         81.2
Name: 2025, dtype: float64
Coal in 2022: 794.1


**What to notice.**

- Aluminum is the big winner: in 2025 its real price was 8 percent of its 1900 level. Better ways of
  producing it, and cheaper electricity, turned it into an everyday material.
- Coal is the only one of the six that cost more in 2025 than in 1900. Its price jumped in 2022, during
  the energy crisis that followed Russia's invasion of Ukraine, and that single jump lifts the
  arithmetic mean far above the geometric one.

**Your turn.** Find the year in which the real price of copper was lowest, and its value in that year.

<details><summary>Hint</summary>

`prices.set_index("year").copper_real_1900_100` gives the copper column with years as labels. Then use
`.idxmin()` for the year and `.min()` for the value, as in the key-numbers cell above.

</details>

In [32]:
# Your turn: try it here

## Figure 6: A slowdown in U.S. economic growth?

**Goal.** Look at how fast U.S. GDP per person has grown, decade by decade, since 1960.

A single year's growth jumps around with booms and recessions. To see the trend, the textbook averages
growth over the *preceding* ten years. In year $t$, with $y$ the real GDP per person,

$$\text{average growth} = 100 \times \frac{\ln y_t - \ln y_{t-10}}{10}\ \text{percent a year}.$$

We load real GDP per person for every year since 1929 and the dates of the U.S. recessions.

In [33]:
gdp = pd.read_csv(DATA_URL + "us_gdp_pc.csv")
recessions = pd.read_csv(DATA_URL + "recessions.csv")
gdp.tail()

,year,gdp_pc
92,2021,64829.5
93,2022,66037.8
94,2023,67413.3
95,2024,68789.3
96,2025,70021.1


`gdp_pc` is real GDP per person in dollars of 2017. In `recessions`, `peak_year` and `trough_year` give the
start and end of each recession as decimal years (1960.25 is April 1960).

> **Python note.** `gdp.copy()` makes a copy of the table, so that adding a column inside the function
> leaves `gdp` itself unchanged. `.shift(window)` moves a column down by `window` rows, so in each row it gives the value
> from `window` years earlier. `np.log` is the natural logarithm. The `for` loop with `zip` hands over one
> (start, end) pair of each recession at a time, and `fig.add_vrect` shades the band between them.

In [34]:
def slowdown_chart(window=10):
    """Figure 6.6: average annual growth of U.S. GDP per person over the preceding `window` years."""
    table = gdp.copy()
    table["growth"] = 100 * (np.log(table.gdp_pc) - np.log(table.gdp_pc.shift(window))) / window
    table = table[table.year >= 1960]
    before = table[table.year <= 2006].growth.mean()
    since = table[table.year >= 2009].growth.mean()
    fig = px.line(table, x="year", y="growth",
                  labels={"year": "Year", "growth": f"{window}-year average growth (%)"},
                  title=f"{window}-year average growth of GDP per person: {before:.1f}% in 1960–2006, "
                        f"{since:.1f}% since 2009")
    fig.update_traces(line=dict(color=BLUE, width=3))
    for start, end in zip(recessions.peak_year, recessions.trough_year):
        fig.add_vrect(x0=start, x1=end, fillcolor=BLUE, opacity=0.12, line_width=0)
    fig.update_layout(yaxis_range=[-0.5, 4.5], xaxis_range=[1959, 2026])
    return fig

> **Python note.** A `Dropdown` is a menu of choices. Here it sets the number of years in the average.

In [35]:
interact(slowdown_chart,
         window=Dropdown(options=[5, 10, 15, 20], value=10, description="years in the average", style=WIDE));

In [36]:
# @title Website version: Figure 6 with a precomputed slider for the number of years in the average
slider_figure(slowdown_chart, "window", [5, 10, 15, 20], start=10, label="years in the average")

**What to notice.**

- From 1960 to 2006 the 10-year average moved between about 1.2 and 3.4 percent, around a fairly
  stable mean of 2.2 percent. After the Great Recession of 2007–2009 it fell below 1 percent.
- The textbook's data end in 2018. With seven more years, the 10-year average has recovered to about
  1.8 percent, still below the pre-2007 mean.
- Shorter windows (5 years) show more ups and downs; longer windows (20 years) smooth them away.

### Key numbers: the slowdown and growth accounting (Table 6.2)

In [37]:
ten = pd.read_csv(DATA_URL + "fig6-6.csv").set_index("year").gdp_pc_growth_10yr
print(f"Mean 1960-2006: {ten.loc[1960:2006].mean():.2f}%   (range {ten.loc[1960:2006].min():.2f} to {ten.loc[1960:2006].max():.2f})")
print(f"Mean 2009-2018: {ten.loc[2009:2018].mean():.2f}%   (lowest: {ten.min():.2f} in {ten.idxmin()})")
print(f"1983: {ten[1983]:.2f}%   1985: {ten[1985]:.2f}%   (the quick rebound after 1982)")
print(f"2019-2025:", ten.loc[2019:2025].round(2).tolist())

Mean 1960-2006: 2.18%   (range 1.24 to 3.36)
Mean 2009-2018: 0.87%   (lowest: 0.70 in 2016)
1983: 1.24%   1985: 2.43%   (the quick rebound after 1982)
2019-2025: [1.64, 1.2, 1.72, 1.75, 1.82, 1.86, 1.82]


What is behind the slowdown? **Growth accounting** splits the growth of output per hour into three parts,
using the production function $Y_t = A_tK_t^{1/3}L_{yt}^{2/3}$ (6.13):

$$\underbrace{g_{Y}-g_{L}}_{\text{output per hour}} = \underbrace{\tfrac{1}{3}(g_{K}-g_{L})}_{\text{capital per hour}} + \underbrace{\tfrac{2}{3}(g_{L_y}-g_{L})}_{\text{labor composition}} + \underbrace{g_{A}}_{\text{TFP}} \qquad (6.15)$$

The Bureau of Labor Statistics publishes an index for each of the four terms. The average growth of an
index between years $a$ and $b$ is $100 \times \ln(X_b/X_a)/(b-a)$.

> **Python note.** `.loc[end]` picks the row of one year. Dividing two rows divides them column by column,
> so one line computes the growth of all four indexes. A dictionary of columns then becomes a table.

In [38]:
accounts = pd.read_csv(DATA_URL + "growth_accounting.csv").set_index("year")


def period_growth(start, end):
    """Average annual growth (percent, log points) of every index from start to end (Table 6.2)."""
    return 100 * np.log(accounts.loc[end] / accounts.loc[start]) / (end - start)


columns = {}
for start, end in [(1948, 2021), (1948, 1973), (1973, 1995), (1995, 2003), (2003, 2021), (2003, 2025)]:
    columns[f"{start}–{end}"] = period_growth(start, end)
table62 = pd.DataFrame(columns)
table62.index = ["Output per hour, Y/L", "Contribution of K/L", "Labor composition", "Contribution of TFP, A"]
table62.round(1)

,1948–2021,1948–1973,1973–1995,1995–2003,2003–2021,2003–2025
"Output per hour, Y/L",2.4,3.3,1.5,3.2,1.9,1.8
Contribution of K/L,0.9,1.0,0.8,1.4,0.8,0.7
Labor composition,0.2,0.2,0.2,0.3,0.3,0.3
"Contribution of TFP, A",1.3,2.1,0.5,1.5,0.8,0.8


The textbook's Table 6.2 for comparison (the last column is new):

In [39]:
book = pd.DataFrame({"1948–2021": [2.4, 0.9, 0.2, 1.2], "1948–1973": [3.3, 1.0, 0.2, 2.1],
                     "1973–1995": [1.5, 0.8, 0.2, 0.5], "1995–2003": [3.2, 1.4, 0.3, 1.5],
                     "2003–2021": [1.8, 0.8, 0.3, 0.7]}, index=table62.index)
book

,1948–2021,1948–1973,1973–1995,1995–2003,2003–2021
"Output per hour, Y/L",2.4,3.3,1.5,3.2,1.8
Contribution of K/L,0.9,1.0,0.8,1.4,0.8
Labor composition,0.2,0.2,0.2,0.3,0.3
"Contribution of TFP, A",1.2,2.1,0.5,1.5,0.7


Where do our numbers differ from the book's? We subtract the two tables (rounded to one decimal) and
check that the three contributions add up to the growth of output per hour, as equation (6.15) says.

In [40]:
print("Our table minus the book's table:")
print(table62[book.columns].round(1) - book)
gap = table62.iloc[1:].sum() - table62.iloc[0]      # contributions minus output per hour, every period
print(f"\nLargest gap between the sum of contributions and output per hour: {gap.abs().max():.3f}")

Our table minus the book's table:
                        1948–2021  1948–1973  1973–1995  1995–2003  2003–2021
Output per hour, Y/L          0.0        0.0        0.0        0.0        0.1
Contribution of K/L           0.0        0.0        0.0        0.0        0.0
Labor composition             0.0        0.0        0.0        0.0        0.0
Contribution of TFP, A        0.1        0.0        0.0        0.0        0.1

Largest gap between the sum of contributions and output per hour: 0.001


Only three cells differ, by 0.1 point each, because the BLS has revised its data since the book was
written: TFP growth over 1948–2021, and output per hour and TFP over 2003–2021. The productivity slowdown after 1973 and the new slowdown after 2003 are both
mostly slowdowns in **TFP growth**, the part that the Romer model attributes to new ideas.

**Your turn.** Add a column for 2019–2025 to the growth-accounting table with `period_growth`. Did TFP grow
faster or slower than in 2003–2021?

<details><summary>Hint</summary>

`period_growth(2019, 2025).round(2)` gives the four numbers. Compare the last one with the TFP value in the
2003–2021 column of `table62`.

</details>

In [41]:
# Your turn: try it here

## Figure 7: The word processor production function

**Goal.** Apply the economics of ideas to software: a large fixed cost and a tiny marginal cost.

This figure comes from a worked exercise in the textbook. A word processor with perfect speech
recognition costs \$100 million to create, and each copy then costs \$1 to distribute. With $X$
million dollars spent, the number of copies (in millions) is

$$Y = X - 100 \quad \text{for } X > 100, \qquad Y = 0 \text{ otherwise.}$$

If the firm sells copies at a price $p$ above the marginal cost of \$1, it **breaks even** when revenue
covers the total cost: $pY = 100 + Y$, so $Y = 100/(p-1)$ million copies.

In [42]:
def word_processor(F=100.0, p=20.0):
    """Figure 6.7: copies (millions) against spending X (millions of $), fixed cost F, $1 per copy."""
    X = np.linspace(0, 210, 211)
    copies = np.maximum(X - F, 0)                    # nothing until the program exists
    breakeven = F / (p - 1)                          # copies needed to cover all costs at price p
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=X, y=copies, name="Production function", line=dict(color=TEAL, width=4)))
    fig.add_trace(go.Scatter(x=[F + breakeven], y=[breakeven], name=f"Break-even point at a price of {p:.0f} dollars",
                             mode="markers", marker=dict(color="black", size=12)))
    fig.update_layout(title=f"At a price of {p:.0f} dollars a copy, the firm breaks even at {breakeven:.2f} million copies",
                      xaxis=dict(title="Input, X (millions of dollars)", range=[0, 210]),
                      yaxis=dict(title="Output, Y (millions of copies)", range=[0, 105]))
    return fig

In [43]:
interact(word_processor,
         F=FloatSlider(value=100, min=50, max=150, step=10, description="fixed cost F (millions of $)", style=WIDE),
         p=FloatSlider(value=20, min=2, max=100, step=1, description="price per copy p ($)", style=WIDE));

In [44]:
# @title Website version: Figure 7 with a precomputed slider for the price per copy
slider_figure(word_processor, "p", [2, 3, 5, 10, 20, 50, 100], start=20, label="price per copy p ($)")

**What to notice.**

- The production function is flat at zero up to \$100 million and then rises one for one. It "curves
  up" from the origin: spending \$200 million gives 100 million copies, but spending half as much gives
  none at all. That is increasing returns.
- At a price of \$20 the firm needs 5.26 million copies to break even; at \$100 it needs only 1.01 million.
- At a price of \$2 the break-even point is at 100 million copies: the firm can survive at a low price
  only if the market is huge. **Scale matters.**

### Key numbers: profits at marginal cost

In [45]:
copies = 1.0                                        # one million copies sold
revenue, cost = 1.0 * copies, 100 + 1.0 * copies    # price = marginal cost = $1; millions of dollars
print(f"Profit at a price of $1: {revenue - cost:.0f} million dollars")
for p in [20, 100]:
    print(f"Break-even sales at p = ${p}: {100 / (p - 1):.2f} million copies")

Profit at a price of $1: -100 million dollars
Break-even sales at p = $20: 5.26 million copies
Break-even sales at p = $100: 1.01 million copies


**Your turn.** The firm can sell 2 million copies. What is the lowest price at which it does not lose
money? Find it with the sliders, then compute it in the empty cell below.

<details><summary>Hint</summary>

Break even means $pY = F + Y$. With $Y = 2$ and $F = 100$, solve for $p$.

</details>

In [46]:
# Your turn: try it here

## Figure 8: Output after a rise in the investment rate

**Goal.** Combine the two models: a Solow economy with capital that sits on top of the Romer
growth in ideas. Then raise the investment rate.

The combined model (textbook Section 6.9) adds capital to the Romer model. Output is
$Y_t = A_t^{\gamma}K_t^{1/3}L_{yt}^{2/3}$ (6.16), capital accumulates as in Chapter 5,
$\Delta K_{t+1} = \bar{s}Y_t - \bar{d}K_t$ (6.17), and ideas, workers and population follow the Romer
equations (6.18)–(6.21). On the balanced growth path,

$$g_y^* = \tfrac{3}{2}\gamma\bar{n} \quad (6.25) \qquad\qquad y_t^* = \left(\frac{\bar{s}}{g_Y^* + \bar{d}}\right)^{1/2} \left(A_t^*\right)^{3\gamma/2}(1-\bar{\ell}) \quad (6.27)$$

where $g_Y^* = g_y^* + \bar{n}$ is the growth of total output. (The textbook prints (6.27) for the case
$\gamma = 1$, where the exponent on $A_t^*$ is $3/2$.) Growth is *faster* than in the Romer
model: ideas raise output, the extra output is partly invested, and the extra capital raises
output again. A higher investment rate $\bar{s}$ raises the **level** of the BGP, not its growth rate.

The function below is the combined model's version of `simulate_romer`. In yearly steps, output per
person on the BGP grows by the factor $(1+\bar{n})^{3\gamma/2}$, which is close to $1 + \tfrac{3}{2}\gamma\bar{n}$.

In [47]:
def simulate_combined(s_old=0.05, s_new=0.05, gamma=2 / 3, n=0.03, d=0.02, z=1.0, l=0.1,
                      change_year=2030, first_year=2000, last_year=2090):
    """Combined Solow-Romer model (equations 6.16-6.21) year by year; s changes to s_new in change_year."""
    years = np.arange(first_year, last_year + 1)
    g_y = (1 + n) ** (1.5 * gamma) - 1              # BGP growth of output per person, yearly steps
    g_Y = (1 + g_y) * (1 + n) - 1                   # BGP growth of total output
    N = 1.0
    A = z * l * N / n                               # BGP stock of ideas, as in the Romer model
    K = (s_old / (g_Y + d)) ** 1.5 * A ** (1.5 * gamma) * (1 - l) * N   # BGP capital, from (6.26)
    y_path, bgp_path = [], []
    for year in years:
        if year < change_year:
            s = s_old
        else:
            s = s_new
        Y = A ** gamma * K ** (1 / 3) * ((1 - l) * N) ** (2 / 3)        # output, equation (6.16)
        y_path.append(Y / N)
        bgp_path.append((s / (g_Y + d)) ** 0.5 * A ** (1.5 * gamma) * (1 - l))   # equation (6.27)
        K = K + s * Y - d * K                       # capital accumulation, equation (6.17)
        A = A + z * l * N                           # new ideas, equation (6.18)
        N = N * (1 + n)                             # population growth, equation (6.21)
    scale = 100 / y_path[0]                         # normalization: y = 100 in the first year
    return years, scale * np.array(y_path), scale * np.array(bgp_path)

In [48]:
def rise_in_s(s_old=0.05, s_new=0.6, scale="ratio"):
    """Figure 6.8: output per person when the investment rate rises from s_old to s_new in 2030."""
    years, y, y_bgp = simulate_combined(s_old=s_old, s_new=s_new)
    years, y_old, y_old_bgp = simulate_combined(s_old=s_old, s_new=s_old)   # the old balanced growth path
    after = years >= 2030
    level = y_bgp[-1] / y_old[-1]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=years[after], y=y_old[after], name="Old BGP", line=dict(color=GRAY, width=3, dash="dash")))
    fig.add_trace(go.Scatter(x=years[after], y=y_bgp[after], name="New BGP", line=dict(color=GREEN, width=3, dash="dash")))
    fig.add_trace(go.Scatter(x=years, y=y, name="Output per person, y", line=dict(color=BLUE, width=3)))
    fig.add_trace(go.Scatter(x=[2093, 2093], y=[y_old[-1], y_bgp[-1]], name=f"Level effect: ×{level:.2f}",
                             mode="lines+markers", line=dict(color="black", width=2)))
    vertical_axis(fig, scale)
    fig.update_layout(title=f"New BGP ×{level:.2f} the old | growth {100 * (y[31] / y[30] - 1):.1f}% in 2031, "
                            f"{100 * (y[-1] / y[-2] - 1):.1f}% in 2090 (BGP: {100 * (y[1] / y[0] - 1):.1f}%)")
    return fig

In [49]:
interact(rise_in_s,
         s_old=FloatSlider(value=0.05, min=0.05, max=0.8, step=0.05, description="old investment rate s̄", style=WIDE),
         s_new=FloatSlider(value=0.6, min=0.05, max=0.8, step=0.05, description="new investment rate s̄'", style=WIDE),
         scale=ToggleButtons(options=["ratio", "linear"], description="vertical scale"));

In [50]:
# @title Website version: Figure 8 with a precomputed slider for the new investment rate
slider_figure(rise_in_s, "s_new", [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8], start=0.6,
              label="new investment rate s̄'")

**What to notice.**

- Before 2030 output per person grows at 3 percent a year, which is $\tfrac{3}{2}\gamma\bar{n}$ with
  $\gamma = 2/3$ and $\bar{n} = 0.03$. In the Romer model with the same parameters it would be only 2 percent.
- In 2030 the investment rate jumps from 0.05 to 0.6. Output does not jump, because the capital stock
  needs time to grow, but growth rises at once to about 26 percent, then slows back toward 3 percent.
- In the long run, output per person is $\sqrt{0.6/0.05} \approx 3.46$ times higher: a level effect,
  just like in the Solow model. The long-run growth rate is still set by ideas.

**Your turn.** Set the old investment rate to 0.6 and the new one to 0.3. Describe the path of output
per person after 2030. Is it ever *falling*?

<details><summary>Hint</summary>

The economy is now *above* its new BGP. The principle of transition dynamics says it grows *slower* than
3 percent for a while. Read the growth rate for 2031 in the title.

</details>

In [51]:
# Your turn: try it here

## Key numbers

This table collects the main numbers of the chapter. Each one is computed with the functions and data
used above.

In [52]:
years, y_fig2, bgp_fig2 = simulate_romer(gamma=0.7, n=0.05)
years, y_romer, bgp_romer = simulate_romer(gamma=2 / 3, n=0.03)
years, y_comb, bgp_comb = simulate_combined()
rows = [
    ["Output when K, L and A all double (equation 6.1)", output_with_ideas(2, 2, 2) / output_with_ideas(1, 1, 1)],
    ["Possible idea paragraphs, 20,000^100: power of 10", 100 * np.log10(20000)],
    ["Doses from the second $2.5 billion (millions)", doses(5.0, F=2.5)],
    ["Romer BGP growth, gamma = 0.7, n = 0.05 (%)", 100 * (y_fig2[1] / y_fig2[0] - 1)],
    ["Level effect of an eightfold rise in z (gamma = 0.6)", 8 ** 0.6],
    ["Level effect of l from 0.03 to 0.35 (gamma = 0.6)", bgp_level(0.35) / bgp_level(0.03)],
    ["Break-even copies at $20 (millions)", 100 / (20 - 1)],
    ["Romer growth, gamma = 2/3, n = 0.03 (%)", 100 * (y_romer[1] / y_romer[0] - 1)],
    ["Combined model growth, same parameters (%)", 100 * (y_comb[1] / y_comb[0] - 1)],
    ["Level effect of s from 0.05 to 0.6", (0.6 / 0.05) ** 0.5],
    ["Commodity prices: fall from 1900 to 2000 (factor)", index[1900] / index[2000]],
    ["U.S. 10-year growth, mean 1960-2006 (%)", ten.loc[1960:2006].mean()],
    ["U.S. 10-year growth, mean 2009-2018 (%)", ten.loc[2009:2018].mean()],
    ["TFP growth, 1948-1973 (%)", table62.loc["Contribution of TFP, A", "1948–1973"]],
    ["TFP growth, 1973-1995 (%)", table62.loc["Contribution of TFP, A", "1973–1995"]],
]
pd.DataFrame(rows, columns=["Quantity", "Value"]).round(2)

,Quantity,Value
0,"Output when K, L and A all double (equation 6.1)",4.00
1,"Possible idea paragraphs, 20,000^100: power of 10",430.10
2,Doses from the second $2.5 billion (millions),250.00
3,"Romer BGP growth, gamma = 0.7, n = 0.05 (%)",3.47
4,Level effect of an eightfold rise in z (gamma ...,3.48
5,Level effect of l from 0.03 to 0.35 (gamma = 0.6),2.93
6,Break-even copies at $20 (millions),5.26
7,"Romer growth, gamma = 2/3, n = 0.03 (%)",1.99
8,"Combined model growth, same parameters (%)",3.00
9,Level effect of s from 0.05 to 0.6,3.46


## Summary

- Romer divides goods into **objects** and **ideas**. Ideas are **nonrival**: they can be used by any
  number of people at once, so there are constant returns to objects and **increasing returns** to objects
  and ideas together.
- Increasing returns come with fixed research costs. If prices equaled marginal cost, nobody could pay
  for research, so innovation needs patents, secrecy, public funding or prizes.
- In the **Romer model**, more people mean more researchers and more ideas, and long-run growth is
  $g_y^* = \gamma\bar{n}$. Changes in research productivity or the research share have **level effects**
  and transition dynamics, but leave long-run growth unchanged.
- The **combined Solow–Romer model** grows at $\tfrac{3}{2}\gamma\bar{n}$: capital amplifies the growth of ideas.
  Investment rates and other Solow parameters set the level of the BGP, and the principle of transition
  dynamics explains why countries grow at different rates for decades.
- The next chapter turns from long-run growth to the **labor market**: wages and unemployment.